<a href="https://colab.research.google.com/github/dhana0020/Deep-Learning/blob/main/vgg_and_google_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To implement VGG and GoogleNet without predefined functions using CNN and compare their performance and modal summary

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from tensorflow.keras import layers


In [ ]:
TRAIN_SAMPLES = 4000
TEST_SAMPLES = 800

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 5


In [ ]:
ds_train = tfds.load(
    'cats_vs_dogs',
    split='train[:4000]',
    as_supervised=True
)

ds_test = tfds.load(
    'cats_vs_dogs',
    split='train[4000:4800]',
    as_supervised=True
)


In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

ds_train = (
    ds_train
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(500)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

ds_test = (
    ds_test
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


VGG

In [ ]:
def build_vgg():
    model = models.Sequential([
        layers.Conv2D(64, (3,3), activation='relu', padding='same',
                      input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.6),
        layers.Dense(1, activation='sigmoid')
    ])
    return model


In [ ]:
vgg_model = build_vgg()
vgg_model.compile(
    optimizer=Adam(0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

vgg_model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)              │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 65536)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │    16,777,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,333,057 (66.12 MB)

 Trainable params: 17,333,057 (66.12 MB)

 Non-trainable params: 0 (0.00 B)

GOOGLE NET



In [ ]:


def inception_block(x, f1, f3_in, f3, f5):

    path1 = layers.Conv2D(f1, (1,1), padding='same', activation='relu')(x)


    path2 = layers.Conv2D(f3_in, (1,1), padding='same', activation='relu')(x)
    path2 = layers.Conv2D(f3, (3,3), padding='same', activation='relu')(path2)


    path3 = layers.Conv2D(f5, (1,1), padding='same', activation='relu')(x)
    path3 = layers.Conv2D(f5, (5,5), padding='same', activation='relu')(path3)


    path4 = layers.MaxPooling2D((3,3), strides=(1,1), padding='same')(x)
    path4 = layers.Conv2D(f1, (1,1), padding='same', activation='relu')(path4)


    return layers.concatenate([path1, path2, path3, path4])




def build_googlenet():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = layers.Conv2D(64, (7,7), strides=2, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

    x = inception_block(x, 32, 64, 16, 16)
    x = inception_block(x, 64, 128, 32, 32)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs, outputs)


In [ ]:
googlenet_model = build_googlenet()
googlenet_model.compile(
    optimizer=Adam(0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

googlenet_model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 64, 64,    │      9,472 │ input_layer_7[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_13    │ (None, 32, 32,    │          0 │ conv2d_19[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 32, 32,    │      4,160 │ max_pooling2d_13… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │      1,040 │ max_pooling2d_13… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_14    │ (None, 32, 32,    │          0 │ max_pooling2d_13… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 32, 32,    │      2,080 │ max_pooling2d_13… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 32, 32,    │      9,232 │ conv2d_21[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 32, 32,    │      6,416 │ conv2d_23[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 32, 32,    │      2,080 │ max_pooling2d_14… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32, 32,    │          0 │ conv2d_20[0][0],  │
│ (Concatenate)       │ 96)               │            │ conv2d_22[0][0],  │
│                     │                   │            │ conv2d_24[0][0],  │
│                     │                   │            │ conv2d_25[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 32, 32,    │     12,416 │ concatenate[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_29 (Conv2D)  │ (None, 32, 32,    │      3,104 │ concatenate[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_15    │ (None, 32, 32,    │          0 │ concatenate[0][0] │
│ (MaxPooling2D)      │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 32, 32,    │      6,208 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 32, 32,    │     36,896 │ conv2d_27[0][0]   │
│                     │ 32)               │            │                 

 Total params: 149,777 (585.07 KB)

 Trainable params: 149,777 (585.07 KB)

 Non-trainable params: 0 (0.00 B)

TrainING

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


history_vgg = vgg_model.fit(
    ds_train,
    validation_data=ds_test,
    epochs=15,callbacks=[early_stop]
)
history_google = googlenet_model.fit(
    ds_train,
    validation_data=ds_test,
    epochs=15,callbacks=[early_stop]
)


Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 0.7814 - loss: 0.4667 - val_accuracy: 0.7600 - val_loss: 0.4834
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 20s 101ms/step - accuracy: 0.8010 - loss: 0.4390 - val_accuracy: 0.7700 - val_loss: 0.4723
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 13s 100ms/step - accuracy: 0.8237 - loss: 0.3917 - val_accuracy: 0.7825 - val_loss: 0.4540
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 20s 100ms/step - accuracy: 0.8478 - loss: 0.3603 - val_accuracy: 0.7688 - val_loss: 0.4772
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 13s 102ms/step - accuracy: 0.8666 - loss: 0.3174 - val_accuracy: 0.7925 - val_loss: 0.4951
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 13s 101ms/step - accuracy: 0.8764 - loss: 0.2875 - val_accuracy: 0.8050 - val_loss: 0.4565
Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 22s 73ms/step - accuracy: 0.5148 - loss: 0.6929 - val_accuracy: 0.5150 - val_loss: 0.6885
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.5321 - loss: 0.68

In [ ]:
print("VGG → Train Loss:", history_vgg.history['loss'][-1])
print("VGG → Val Loss:", history_vgg.history['val_loss'][-1])
print("VGG → Val Accuracy:", history_vgg.history['val_accuracy'][-1])




VGG → Train Loss: 0.28522661328315735
VGG → Val Loss: 0.45647528767585754
VGG → Val Accuracy: 0.8050000071525574


In [ ]:
print("GoogLeNet → Train Loss:", history_google.history['loss'][-1])
print("GoogLeNet → Val Loss:", history_google.history['val_loss'][-1])
print("GoogLeNet → Val Accuracy:", history_google.history['val_accuracy'][-1])

GoogLeNet → Train Loss: 0.6205894351005554
GoogLeNet → Val Loss: 0.618699848651886
GoogLeNet → Val Accuracy: 0.643750011920929
